# Likelihood Ratio Test (LRT) demo.

When fitting models of different sizes (number of free parameters), changing to a model with more parameters will generally fit with better (higher) likelihood on the same dataset, but how do we know the improved likelihood represents the model becoming closer to the true distribution and not overfitting because the model is more complex?

With statistical hypothesis testing, more specifically, using the Likelihood Ratio Test:
 * Let the null hypothesis $H_0$ be that our data are generated by model $M_0$ with $p_0$ parameters and the likelihood of that model is $L_0$.
 * Let the alternate hypothesis be that it came from model $M_{alt}$ with $p_{alt}$ parameters and that it yields likelihood $L_{alt}$ when fit to the same data.
 * Then the LRT statistic $ LRT = 2 * (\log L_{alt} - \log L_0)$ has a $\chi^2(d)$ distribution**, where the degrees of freedom is the difference in the number of parameters, $d = p_{alt} - p_0$.
 * The p-value for LRT is the probability of model $M_0$ generating data at least as improbable as the given dataset, compared to $M_{alt}$, i.e.:
    * A high p-value means the alternate model does not provide much additional explanatory power and we should not reject $H_0$, inversely,
    * A low p-value means the dataset looks extremely unlikely under the null hypothesis when compared to the alternate so we should reject $H_{0}$, preferring $H_{alt}$.
      
**NOTE: this is subject to the requirements of [Wilk's theorem](https://en.wikipedia.org/wiki/Wilks%27_theorem):

* The models must be "nested" (the simpler model is a special case of the more complex model), which is satisfied in our demo, but the
* true parameters must not lie on the boundary of the parameter space of the more complex model, which is the case in our GMM comparison.
  
This means our P-values will be approximate, but they can be made very accurate by increasing the number of points.

## Demo
   
This demo generates synthetic data from a 2-component gaussian mixture model (GMM), fits three models of different sizes, and compares the likelihood ratio statistics between them to determin which is "best".  The models are:
* "1-comp," a single Gaussian (underparameterized),
* "2-comp," a 2-component Gaussian mixture ("just right"), and
* "3-comp," a 3-component Gaussian mixture (overparameterized).

The demo shows three plots, one for each model, showing its theoretical distribution (PDF) over a hisgotram of the dataset.

The three pairwise model comparisons are:
* $H_0$: 1-comp, $H_{alt}$: 2-comp (comparing a single gaussian model to a 2-component GMM)
* $H_0$: 2-comp, $H_{alt}$: 3-comp (comparing the 2-component GMM to a 3-component GMM)
* $H_0$: 1-comp, $H_{alt}$: 3-comp (comparing the single component model to the 3-component model)

For each comparison, three sample LRT values are printed for $H_0$ rejection, at confidence levels of 90%, 95%, and 99% in addition to the computed LRT statistic and the p-value (probability of getting a statistic value as or more extreme under $H_0$).

### Observations:

* Moving the data clusters together (mean_dist --> 0) means the 1-component gaussian is the best model:
* Making them well-separated means the 2-component model will have the best fit.
  
In all cases, the relative *likelihoods* will remain in the order $L(GMM(1)) < L(GMM(2)) < L(GMM(3))$, but the LRT / p-value should indicate clearly one of these two states (or somewhere in between, etc.)
### Interaction

User controls these demo parameters with sliders:
 * `N_points` - number of sample points generated
 * `mean_dist` - distance between gaussian components used to generate data (each has unit SD for simplicity).
 * `N_bins` - number of histogram bins to adjust for clarity.
### Installation / Requirements

The notebook requires common scientific Python packages: `numpy`, `scipy`, `matplotlib`, `scikit-learn`, and `ipywidgets` for interactive sliders. If you don't have them, run:
```
pip install numpy scipy matplotlib scikit-learn ipywidgets jupyterlab_widgets
```
You may have to also install jupyterlab-manager:
```
jupyter labextension install @jupyter-widgets/jupyterlab-manager
```
Then run the cells below. The main demo cell builds the interactive interface and should render inline in JupyterLab / classic Jupyter Notebook.


In [48]:
# Cell 2: Full interactive demo code
# Paste this into a code cell and run.

import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
from scipy import stats
from scipy.stats import chi2, norm
from sklearn.mixture import GaussianMixture
import ipywidgets as widgets
from IPython.display import display, clear_output

# -------------------------
# Utility / helper functions
# -------------------------

def sample_true_distribution(N, mean_dist, rng=None):
    """Sample N points from a 2-component equal-weight Gaussian mixture (unit SD).
       Means are at -mean_dist/2 and +mean_dist/2 so center is zero.
    """
    if rng is None:
        rng = np.random.RandomState()
    means = np.array([-mean_dist/2.0, mean_dist/2.0])
    components = rng.choice([0, 1], size=N, p=[0.5, 0.5])
    x = rng.normal(loc=means[components], scale=1.0, size=N)
    return x

def fit_gmm(X, k, random_state=0, n_init=5):
    """Fit a k-component GaussianMixture, return fitted model and total log-likelihood."""
    gm = GaussianMixture(n_components=k, covariance_type='full', random_state=random_state, n_init=n_init)
    gm.fit(X.reshape(-1,1))
    # gm.score returns average log-likelihood per sample; multiply to get total ll
    ll = gm.score(X.reshape(-1,1)) * len(X)
    return gm, ll

def gmm_pdf(gm, xs):
    """Return pdf values for a fitted sklearn GMM over xs (1D)."""
    pdf = np.zeros_like(xs, dtype=float)
    weights = gm.weights_
    means = gm.means_.flatten()
    # covariance array shape handling
    covs = gm.covariances_
    # covs may be shape (k, 1, 1) or (k,) depending on sklearn version
    if covs.ndim == 3:
        variances = covs.reshape(covs.shape[0], -1)[:,0]
    else:
        variances = covs
    stds = np.sqrt(variances)
    for w, m, s in zip(weights, means, stds):
        pdf += w * stats.norm.pdf(xs, loc=m, scale=s)
    return pdf

def gmm_components(gm):
    """Return list of (weight, mean, std) for components of a fitted GMM."""
    weights = gm.weights_
    means = gm.means_.flatten()
    covs = gm.covariances_
    if covs.ndim == 3:
        variances = covs.reshape(covs.shape[0], -1)[:,0]
    else:
        variances = covs
    stds = np.sqrt(variances)
    return list(zip(weights, means, stds))

def gmm_param_count(k):
    """Count free parameters for a 1D GMM: (k-1) component priors + k means + k variances = 3k - 1"""
    return 3*k - 1

def chi2_thresholds(df):
    return {'90%': chi2.ppf(0.90, df), '95%': chi2.ppf(0.95, df), '99%': chi2.ppf(0.99, df)}

# -------------------------
# Persistent state & widgets
# -------------------------

# Keep the currently sampled data here so resampling doesn't change sliders
state = {
    'X': None,
    'rng': np.random.RandomState()  # independent rng for resampling
}

# Create widgets (using kwargs explicitly)
resample_button = widgets.Button(description="Resample", button_style='primary', tooltip="Resample synthetic data")
N_points_slider = widgets.IntSlider(value=1500, min=50, max=5000, step=50, description="N_points", continuous_update=True)
mean_dist_slider = widgets.FloatSlider(value=6.0, min=0.0, max=8.0, step=0.1, description="mean_dist", continuous_update=True)
N_bins_slider = widgets.IntSlider(value=40, min=5, max=200, step=1, description="N_bins", continuous_update=True)

controls = widgets.HBox([resample_button, N_points_slider, mean_dist_slider, N_bins_slider])

# Output area for plots
out = widgets.Output(layout=widgets.Layout(border='1px solid lightgray'))

# -------------------------
# Update / draw function
# -------------------------

def update(resample=False):
    """Recompute data (optionally resample), fit models, and redraw the entire 2x2 grid."""
    # Resample data if requested or if we have no data yet
    if resample or state['X'] is None:
        state['X'] = sample_true_distribution(N_points_slider.value, mean_dist_slider.value, rng=state['rng'])
    X = state['X']
    
    # Fit models (use fixed random_state seeds to reduce EM variability across updates)
    g1, ll1 = fit_gmm(X, 1, random_state=0, n_init=5)
    g2, ll2 = fit_gmm(X, 2, random_state=1, n_init=5)
    g3, ll3 = fit_gmm(X, 3, random_state=2, n_init=5)
    
    # Compute LRT statistics (2 * (ll_alt - ll_null))
    LRT_12 = 2.0 * (ll2 - ll1)
    LRT_23 = 2.0 * (ll3 - ll2)
    LRT_13 = 2.0 * (ll3 - ll1)
    
    # Degrees of freedom (difference in parameter counts)
    df_12 = gmm_param_count(2) - gmm_param_count(1)   # typically 3
    df_23 = gmm_param_count(3) - gmm_param_count(2)   # typically 3
    df_13 = gmm_param_count(3) - gmm_param_count(1)   # typically 6
    
    # 1 - p-values (chi-square approximation; shown for pedagogical reasons)
    p12 = 1.0 - chi2.cdf(LRT_12, df_12)
    p23 = 1.0 - chi2.cdf(LRT_23, df_23)
    p13 = 1.0 -chi2.cdf(LRT_13, df_13)
    
    np12 = chi2.cdf(LRT_12, df_12)
    np23 = chi2.cdf(LRT_23, df_23)
    np13 = chi2.cdf(LRT_13, df_13)
    
    # χ² critical values
    thr12 = chi2_thresholds(df_12)
    thr23 = chi2_thresholds(df_23)
    thr13 = chi2_thresholds(df_13)
    
    # Plot area
    with out:
        clear_output(wait=True)
        plt.close('all')
        fig, axs = plt.subplots(2, 2, figsize=(12, 9))
        ax1, ax2 = axs[0]
        ax3, ax4 = axs[1]
        
        # x-range for curve plotting (pad edges)
        xmin, xmax = X.min() - 1.5, X.max() + 1.5
        xs = np.linspace(xmin, xmax, 800)
        
        # --- 1-component plot (top-left) ---
        ax1.hist(X, bins=N_bins_slider.value, density=True, alpha=0.85, edgecolor='w', facecolor='k')
        pdf1 = gmm_pdf(g1, xs)
        ax1.plot(xs, pdf1, lw=2, label='1-comp PDF')
        ax1.set_title(f'1-component Gaussian — LL = {ll1:.2f}')
        ax1.set_ylabel('Density')
        ax1.legend()
        
        # --- 2-component plot (top-right) ---
        ax2.hist(X, bins=N_bins_slider.value, density=True, alpha=0.85, edgecolor='w', facecolor='k')
        pdf2 = gmm_pdf(g2, xs)
        ax2.plot(xs, pdf2, lw=2, label='2-comp PDF')
        # dashed individual components
        for comp_ind,(w,m,s) in enumerate(gmm_components(g2)):
            ax2.plot(xs, w * norm.pdf(xs, loc=m, scale=s), '--', linewidth=2, label='comp %i' % (comp_ind, ))
        ax2.set_title(f'2-component GMM — LL = {ll2:.2f}')
        ax2.legend()
        
        # --- 3-component plot (bottom-left) ---
        ax3.hist(X, bins=N_bins_slider.value, density=True, alpha=0.85, edgecolor='w', facecolor='k')
        pdf3 = gmm_pdf(g3, xs)
        ax3.plot(xs, pdf3, lw=2, label='3-comp PDF')
        for comp_ind,(w,m,s) in enumerate(gmm_components(g3)):
            ax3.plot(xs, w * norm.pdf(xs, loc=m, scale=s), '--', linewidth=2, label='comp %i' % (comp_ind, ))
        ax3.set_title(f'3-component GMM — LL = {ll3:.2f}')
        ax3.set_xlabel('x')
        ax3.set_ylabel('Density')
        ax3.legend()
        
        # --- Text cell (bottom-right) ---
        ax4.axis('off')
        text = (
            f"Confidence levels for rejecting H_0 given LRT:\n\n"
            f"  H_0: 1-comp, H_alt: 2-components:\n"
            f"       reject w/ 90% if LRT >= {thr12['90%']:6.3f},\n"
            f"                 95% if LRT >= {thr12['95%']:6.3f},\n"
            f"                 99% if LRT >= {thr12['99%']:6.3f}.\n"
            f"       LRT: {LRT_12:.3f} (p-value (χ², df={df_12}) = {p12:.6g})\n\n"
        
            f"  H_0: 2-comp, H_alt: 3-components:\n"
            f"       reject w/ 90% if LRT >= {thr23['90%']:6.3f},\n"
            f"                 95% if LRT >= {thr23['95%']:6.3f},\n"
            f"                 99% if LRT >= {thr23['99%']:6.3f}.\n"
            f"       LRT: {LRT_23:.3f} (p-value (χ², df={df_23}) = {p23:.6g})\n\n"

            f"  H_0: 1-comp, H_alt: 3-components:\n"
            f"       reject w/ 90% if LRT >= {thr13['90%']:6.3f},\n"
            f"                 95% if LRT >= {thr13['95%']:6.3f},\n"
            f"                 99% if LRT >= {thr13['99%']:6.3f}.\n"
            f"       LRT: {LRT_13:.3f} (p-value (χ², df={df_13}) = {p13:.6g})\n\n"
 
            f"(Note: χ² approximation / Wilks' theorem does not \nstrictly apply to GMMs — p-values are illustrative.)"
        )
        # Use monospace and preserve whitespace
        ax4.text(0.01, 0.99, text, va='top', ha='left', family='monospace', fontsize=10)

        for ax in [ax1, ax2, ax3]:
            ax.set_xlim(xmin,xmax)
        
        plt.tight_layout()
        display(fig)
        plt.close(fig)  # avoid duplicate inline rendering

# -------------------------
# Event handlers
# -------------------------

def on_resample_clicked(btn):
    # Resample new dataset using current slider params and redraw
    state['X'] = sample_true_distribution(N_points_slider.value, mean_dist_slider.value, rng=state['rng'])
    update(resample=False)  # full redraw with new data

# Link sliders to update (all changes force full redraw)
def on_slider_change(change):
    # If the slider value changed, we recompute fits on the current data
    update(resample=True)

resample_button.on_click(on_resample_clicked)
N_points_slider.observe(on_slider_change, names='value')
mean_dist_slider.observe(on_slider_change, names='value')
N_bins_slider.observe(on_slider_change, names='value')

# -------------------------
# Initial draw & display
# -------------------------
# Create an initial sample so that the first draw shows data
state['X'] = sample_true_distribution(N_points_slider.value, mean_dist_slider.value, rng=state['rng'])
update(resample=False)

# Display controls and output area
display(controls, out)


Output(layout=Layout(border_bottom='1px solid lightgray', border_left='1px solid lightgray', border_right='1px…

## Notes and caveats
- The Likelihood Ratio Test (LRT) p-values are computed using a chi-square approximation (Wilks). This is not strictly valid for Gaussian Mixture Models because regularity conditions fail; the p-values are shown for pedagogical/demonstration purposes only.
- Prefer information criteria (AIC, BIC) for model comparison in mixture contexts; they are also displayed.
- If you want to make the randomness fully reproducible across slider updates, replace the random generation with a fixed RNG seed and avoid regenerating the mixture component assignments on each update.
